# AI 201 — Week 2 Demo: Concert Alert Agent

We're building a 3-tool agent that:
1. Searches for upcoming shows for artists you follow
2. Checks whether tickets are still available
3. Drafts a text message to send to friends when tickets are found

The agent decides which tool to call, when to call it, and what to do when something goes wrong — sold-out shows, artists with no upcoming dates.

**What we're writing live:**
- The tool definitions (how we describe the tools to the model)
- The agent loop (the ReAct-style planning loop that runs until the job is done)

Everything else — the tools themselves, the concert data, state tracking — is already built.

In [ ]:
# Cell 1 — Setup (run first, then minimize)
from dotenv import load_dotenv
load_dotenv()

import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from tools import TOOL_REGISTRY
from utils.llm import LLMClient
from utils.state import AgentState

llm = LLMClient()
print(f'Ready. Model: {llm.model}')
print(f'Tools available: {list(TOOL_REGISTRY.keys())}')

---
## Part 1: Test Each Tool in Isolation

Before writing the agent, let's see what the tools actually do — and verify they work independently.

This order matters: **test tools in isolation before connecting them to a loop.** An agent built on three untested tools is three times harder to debug than three tools you've already verified. If a tool returns the wrong shape or handles failure badly, you want to know now — not after the loop is running.

The agent doesn't call these directly — it *describes* them to the LLM, and the LLM decides when to use them. But they're just Python functions, and we can run them directly right now.

In [ ]:
# ── DEMO MOMENT 1A — Test Tool 1 in isolation ──────────────────────────────
# Run search_shows directly with a known artist before wiring it to the loop.
# Ask students: "What does the return value look like? What fields are in it?"
# They'll need to know the shape to understand what the agent receives later.
# ───────────────────────────────────────────────────────────────────────────
from tools import search_shows, check_tickets, draft_message

result = search_shows('Tyler, the Creator')
print('search_shows("Tyler, the Creator"):')
print(json.dumps(json.loads(result), indent=2))
print()
print('Note the shape: the agent will parse this JSON to get show IDs for check_tickets.')

In [ ]:
# ── DEMO MOMENT 1B — Test Tool 2 in isolation ──────────────────────────────
# Test check_tickets with a known show_id — verify the happy path.
# Ask students: "What field would the agent check to know if tickets are available?"
# ───────────────────────────────────────────────────────────────────────────
result = check_tickets('SHOW001')
print('check_tickets("SHOW001") — happy path:')
print(json.dumps(json.loads(result), indent=2))

In [ ]:
# ── DEMO MOMENT 1C — Test the failure case in isolation ─────────────────────
# Test check_tickets with a sold-out show before the loop ever runs.
# This is the isolation testing payoff: you know exactly what the agent
# will receive in this case, so when the loop runs you can reason about it.
#
# Ask students: "What should the agent do when it sees this response?"
# Ask: "How does this response differ from the available-tickets case?"
# ────────────────────────────────────────────────────────────────────────────

# SHOW003 = Sabrina Carpenter at MSG — sold out
result = check_tickets('SHOW003')
print('check_tickets("SHOW003") — Sabrina Carpenter at MSG (sold out):')
print(json.dumps(json.loads(result), indent=2))
print()
print('The agent will read this exact JSON. Its behavior depends on the system prompt')
print('and the tool description — not on any "if sold out" logic we write in the loop.')

---
## Part 2: Write the Tool Definitions

The tools are Python functions. But the LLM can't call Python functions — it can only read and write text.

So we describe each tool to the model as a JSON schema: what's it called, what does it do, and what arguments does it take. The model reads this and decides when to use each tool.

**✍️ LIVE CODE — write one tool definition together, then fill in the other two:**

In [ ]:
# ── INSTRUCTOR: Write this live. ────────────────────────────────────────────
#
# Start by writing the search_shows definition together.
# Ask students before writing: "What does the model need to know about a tool?"
#   → What it's called (name)
#   → What it does (description — this is what the model reads to decide when to use it)
#   → What arguments to pass (parameters schema)
#
# Once search_shows is written, point to the other two and say:
#   "Same pattern — different name, description, and parameters."
# Then fill them in. Students should be able to predict the structure.
#
# Key talking point: "We're not calling these tools. We're describing them.
# The model decides when to call them based on these descriptions."
# ────────────────────────────────────────────────────────────────────────────

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_shows",
            "description": "Search for upcoming concerts for a given artist. Call this first for any artist before checking tickets.",
            "parameters": {
                "type": "object",
                "properties": {
                    "artist_name": {
                        "type": "string",
                        "description": "The name of the artist to search for",
                    }
                },
                "required": ["artist_name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_tickets",
            "description": "Check ticket availability for a specific show using its show_id. Only call this after getting a show_id from search_shows.",
            "parameters": {
                "type": "object",
                "properties": {
                    "show_id": {
                        "type": "string",
                        "description": "The unique show identifier returned by search_shows",
                    }
                },
                "required": ["show_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "draft_message",
            "description": "Draft a text message to send to friends about a concert with available tickets. Only call this when tickets are confirmed available.",
            "parameters": {
                "type": "object",
                "properties": {
                    "artist_name": {"type": "string"},
                    "venue":       {"type": "string"},
                    "city":        {"type": "string"},
                    "date":        {"type": "string", "description": "Date in YYYY-MM-DD format"},
                    "price_range": {"type": "string"},
                    "friends":     {"type": "array", "items": {"type": "string"}, "description": "Names of friends to alert"},
                },
                "required": ["artist_name", "venue", "city", "date", "price_range"],
            },
        },
    },
]

print(f'{len(TOOLS)} tool definitions ready.')
print('Tool names:', [t["function"]["name"] for t in TOOLS])

---
## Part 3: Write the Agent Loop

Now the heart of the agent. The loop:
1. Sends the conversation to the LLM (with the tool definitions)
2. If the model wants to call a tool → executes it, appends the result, repeats
3. If the model is done → returns the final answer

This is the ReAct loop. Every agent you'll ever build is a version of this.

**✍️ LIVE CODE — write this together:**

In [ ]:
# ── INSTRUCTOR: Write this live. ────────────────────────────────────────────
#
# Ask students before writing: "What are the three steps in the loop?"
#   1. Ask the model what to do next
#   2. Did it want to call a tool? → run the tool, give it the result
#   3. Did it give a final answer? → we're done
#
# SYSTEM PROMPT — narrate this as you write it:
#   "This is where we define the agent's decision rules. Notice it says
#    'search for upcoming shows' — not 'you can search if you want.' The
#    specificity is what makes the agent call the tool instead of answering
#    from memory. Compare this to 'be a helpful concert assistant' and think
#    about how differently the model would behave."
#
# KEY MOMENTS TO CALL OUT as you write the loop body:
#
#   (1) messages.append(msg) — first append:
#       "We store the model's message before doing anything else.
#        This is the assistant message — it contains the tool call request
#        and the 'id' that ties the request to its result."
#
#   (2) tool_call.id — when building the tool result message:
#       "See this tool_call.id? The API uses it to match this result back
#        to the assistant message that requested it. If the assistant message
#        isn't already in the list before we append this, the API has no
#        record of the request — it'll error or produce garbage output.
#        That's why we always append the assistant message first."
#
#   (3) The stop condition (if not msg.tool_calls):
#       "The loop exits when the model stops calling tools and gives a
#        final answer. If a tool keeps returning empty results and the model
#        keeps retrying, this loop runs forever — max_steps is the safety valve."
#
# Target: ~20 lines. Should take about 5 minutes including narration.
# ────────────────────────────────────────────────────────────────────────────

def run_agent(user_request: str, state: AgentState, max_steps: int = 20) -> str:
    """
    Run the concert alert agent until it completes or hits the step limit.

    Parameters
    ----------
    user_request : str
        What the user asked for.
    state : AgentState
        Tracks what the agent has done (checked artists, found tickets, drafted messages).
    max_steps : int
        Safety limit on tool calls — prevents infinite loops.

    Returns
    -------
    str
        The agent's final text response.
    """
    # ── SYSTEM PROMPT ────────────────────────────────────────────────────────
    # This defines the agent's decision rules — not just its personality.
    # "search for upcoming shows" (not "you can search") tells the model to
    # always use the tool rather than answering from training data.
    # ─────────────────────────────────────────────────────────────────────────
    system_prompt = (
        "You are a concert alert agent. For each artist the user mentions, "
        "search for upcoming shows, check ticket availability, and draft a "
        "friend-alert message only if tickets are available. "
        "If shows are sold out or don't exist, report that clearly. "
        "Check every artist — don't stop after the first one."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_request},
    ]

    for step in range(max_steps):
        print(f"  [step {step + 1}] Asking model what to do next...")
        response = llm.chat(messages, tools=TOOLS)
        msg = response.choices[0].message

        # ── APPEND THE ASSISTANT MESSAGE FIRST ──────────────────────────────
        # This message contains the tool call request AND the 'id' that the
        # tool result will reference. It must be in the list before the result.
        # Appending the result first causes an API error — the reference breaks.
        # ─────────────────────────────────────────────────────────────────────
        messages.append(msg)

        if not msg.tool_calls:
            # No tool calls = the model has a final answer — we're done
            print(f"  [step {step + 1}] Agent finished.")
            return msg.content

        # Execute each tool the model requested
        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"  [step {step + 1}] → calling {tool_name}({tool_args})")

            tool_fn = TOOL_REGISTRY.get(tool_name)
            if tool_fn:
                tool_result = tool_fn(**tool_args)
                state.tool_call_count += 1
            else:
                tool_result = json.dumps({"error": f"Unknown tool: {tool_name}"})

            # ── APPEND THE TOOL RESULT ───────────────────────────────────────
            # tool_call_id must match the 'id' in the assistant message above.
            # The API uses this to link this result to the request that made it.
            # ──────────────────────────────────────────────────────────────────
            messages.append({
                "role":         "tool",
                "tool_call_id": tool_call.id,   # ← matches msg.tool_calls[n].id
                "content":      tool_result,
            })

    return "Agent hit the step limit without finishing."


print('run_agent() defined. Ready to run.')

---
## Part 4: Run the Agent

### Scenario 1 — Happy path: artist with available tickets

In [ ]:
# ── DEMO MOMENT 2 ──────────────────────────────────────────────────────────
# Run the agent on Tyler, the Creator — available tickets.
# Watch the step log as it runs. Ask students:
#   "What did the model call first? Why that order?"
#   "How many steps did the loop take?"
# ───────────────────────────────────────────────────────────────────────────

state = AgentState(artists_requested=["Tyler, the Creator"])

print('Running agent for: Tyler, the Creator\n')
answer = run_agent(
    user_request="Check for upcoming Tyler, the Creator shows and alert my friends Jordan and Maya if there are tickets.",
    state=state,
)
print(f'\nFinal answer:\n{answer}')

### Scenario 2 — Failure handling: sold-out show

In [ ]:
# ── DEMO MOMENT 3 ──────────────────────────────────────────────────────────
# Run the agent on Sabrina Carpenter — all shows sold out.
# Ask students: "We didn't write any 'if sold out' logic. How does the agent
# know what to do?" (It read the tool result and adapted.)
# Ask: "Is this the behavior you'd want in production?"
# ───────────────────────────────────────────────────────────────────────────

state2 = AgentState(artists_requested=["Sabrina Carpenter"])

print('Running agent for: Sabrina Carpenter\n')
answer2 = run_agent(
    user_request="Check for upcoming Sabrina Carpenter shows and alert my friends if there are tickets.",
    state=state2,
)
print(f'\nFinal answer:\n{answer2}')

### Scenario 3 — Multiple artists + the 'no shows' case

In [ ]:
# ── DEMO MOMENT 4 ──────────────────────────────────────────────────────────
# Run the agent on multiple artists at once, including Frank Ocean (no shows).
# Ask students: "What has the state tracker seen by the end of this?"
# Show state.summary() after it finishes.
# Ask: "What would go wrong if we didn't track state across tool calls?"
# ───────────────────────────────────────────────────────────────────────────

artists = ["Chappell Roan", "Frank Ocean", "Doechii"]
state3 = AgentState(artists_requested=artists)

print(f'Running agent for: {artists}\n')
answer3 = run_agent(
    user_request=(
        "I want to know about upcoming shows for Chappell Roan, Frank Ocean, and Doechii. "
        "Alert my friends Alex and Sam for any show with available tickets."
    ),
    state=state3,
)
print(f'\nFinal answer:\n{answer3}')
print(f'\n--- Agent State Summary ---')
print(state3.summary())

---
## Part 5: Observe the Loop

Run this cell to re-run Scenario 3 with verbose output — every message in the conversation history, including tool calls and tool results. This shows students exactly what the model sees on each step.

In [ ]:
# ── DEMO MOMENT 5 ──────────────────────────────────────────────────────────
# Show the full message history after a run — this is what the model sees.
# Ask: "Why do we keep the whole conversation history in memory?"
#   → The model has no memory between calls. The history IS the memory.
# Ask: "What would happen if we only sent the last message each time?"
# ───────────────────────────────────────────────────────────────────────────

# Re-run a short version with message history capture
messages_log = [
    {"role": "system", "content": "You are a concert alert agent. Search for shows, check tickets, draft messages if available."},
    {"role": "user",   "content": "Check upcoming Doechii shows and alert my friend Sam if tickets are available."},
]

for step in range(10):
    response = llm.chat(messages_log, tools=TOOLS)
    msg = response.choices[0].message
    messages_log.append(msg)

    if not msg.tool_calls:
        break

    for tool_call in msg.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        tool_result = TOOL_REGISTRY[tool_name](**tool_args)
        messages_log.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result,
        })

print(f'Total messages in history: {len(messages_log)}\n')
for i, m in enumerate(messages_log):
    role = m["role"] if isinstance(m, dict) else m.role
    if role == 'tool':
        content = json.dumps(json.loads(m['content']), indent=2) if isinstance(m, dict) else ''
        print(f'[{i}] TOOL RESULT: {content[:300]}...' if len(content) > 300 else f'[{i}] TOOL RESULT: {content}')
    elif role == 'assistant':
        content = m if isinstance(m, dict) else m
        has_calls = bool(getattr(content, 'tool_calls', None) if not isinstance(content, dict) else content.get('tool_calls'))
        tc = getattr(msg, 'tool_calls', []) if i == len(messages_log) - 2 else []
        print(f'[{i}] ASSISTANT: {"[tool call requested]" if has_calls else getattr(content, "content", str(content))[:200]}')
    else:
        content_str = m.get('content', '') if isinstance(m, dict) else getattr(m, 'content', '')
        print(f'[{i}] {role.upper()}: {str(content_str)[:200]}')
    print()